In [0]:
import requests
import json
password = dbutils.secrets.get(scope="topdesk-scope", key="topdeskapi")
username = "topdeskapi"

url = "https://elixir.msf.org/tas/api/incidents/timeregistrations?fields=timeSpent,operator.id,creationDate,parent.id,notes"

resp = requests.get(url, auth=(username, password))

data = resp.json()

# Extract the actual records
records = data.get('results',[])

In [0]:
data

In [0]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def fetch_time_registrations(username: str, password: str):
    base_url = (
        "https://elixir.msf.org/tas/api/incidents/timeregistrations" 
        "?fields=timeSpent,operator.id,creationDate,parent.id,notes"
    )

    all_data = []
    url = base_url

    # Configure retry strategy (helps speed when transient failures occur)
    retries = Retry(
        total=3,
        backoff_factor=0.3,
        status_forcelist=(500, 502, 503, 504),
        allowed_methods=("GET",),
    )

    adapter = HTTPAdapter(pool_connections=10, pool_maxsize=10, max_retries=retries)

    with requests.Session() as session:
        session.auth = (username, password)
        session.mount("https://", adapter)

        get = session.get  # local binding for speed
        
        page = 0
        while url:
            resp = get(url, timeout=(3, 10))  # (connect, read)

            if resp.status_code != 200:
                raise RuntimeError(f"API error {resp.status_code}: {resp.text[:200]}")

            payload = resp.json()

            results = payload.get("results")
            if results:
                all_data.extend(results)

            page += 1
            print(f"Fetched page {page}, records so far: {len(all_data)}")


            url = payload.get("next")

    return all_data

In [0]:
password = dbutils.secrets.get(scope="topdesk-scope", key="topdeskapi")
username = "topdeskapi"
all_data = fetch_time_registrations(username, password)

In [0]:
def fetch_time_registrations():
    url = "https://elixir.msf.org/tas/api/incidents/timeregistrations?fields=timeSpent,operator.id,creationDate,parent.id,notes"
    all_data = []
    with requests.Session() as session:
        while url:
            resp = session.get(url, auth=(username, password), timeout = 10)

            if resp.status_code != 200:
                print(f"Error: {resp.status_code}")
                break

            data = resp.json()
            results = data.get("results", [])
            all_data.extend(results)
            url = data.get("next")
    return all_data

In [0]:
records = fetch_time_registrations()


In [0]:
records

In [0]:
tr_id = '69a3b7cf-92ab-4c99-976d-754b86d773ff'
url = "https://elixir.msf.org/tas/api/incidents/timeregistrations"
url_v2 = f"{url}/{tr_id}"
resp = requests.get(url_v2, auth=(username, password))
print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

In [0]:
#====================================================================
# CONFIG
#====================================================================
REQUEST_TIMEOUT = 30
dbutils.widgets.text("env", "brz")
ENV = dbutils.widgets.get("env")
CATALOG = "elixir"

SCHEMA = "meta"
TR_BASE_URL = "https://elixir.msf.org/tas/api/incidents/timeregistrations"
TR_PIPELINE_NAME = "topdesk_incident_timeregistrations"
TR_BRONZE_TABLE = f"{CATALOG}.{ENV}.todesk_incident_timeregistration"
TR_STATE_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_state"


import requests
import time
import json
from datetime import datetime, timezone

# On the very first run, the pipeline performs a FULL load; all subsequent runs switch to INCREMENTAL mode.

In [0]:
dbutils.widgets.text("env", "brz")
ENV = dbutils.widgets.get("env")
CATALOG = "elixir"

SCHEMA = "meta"


In [0]:
# ============================================================
# TOPdesk Incident Ingestion Pipeline (Production Safe)
# ============================================================

import requests
import time
import json
from datetime import datetime, timezone
from typing import Optional, List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import Row

# ================= SECRETS =================

password = dbutils.secrets.get(scope="topdesk-scope", key="topdeskapi")
username = "topdeskapi"

# ================= CONFIG =================

CATALOG = "elixir"
BRONZE_SCHEMA = "brz"
META_SCHEMA = "meta"

#BASE_URL = "https://elixir.msf.org/tas/api/incidents"


AUTH = (username, password)

PAGE_SIZE = 100
MAX_WORKERS = 10
MAX_RETRIES = 3
REQUEST_TIMEOUT = 30

LOAD_TYPE = "FULL"   # "FULL" or "INCREMENTAL"

# PIPELINE_NAME = "topdesk_incidents"

# BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_incidents_raw"
STATE_TABLE  = f"{CATALOG}.{META_SCHEMA}.timeregistrations_state"
BASE_URL = "https://elixir.msf.org/tas/api/incidents/timeregistrations?fields=timeSpent,operator.id,creationDate,parent.id,notes"
PIPELINE_NAME = "topdesk_incident_timeregistrations"
BRONZE_TABLE = "elixir.brz.topdesk_timeregistrations_raw"
# ==========================================


# ============================================================
# STATE MANAGEMENT
# ============================================================

def get_last_run() -> Optional[str]:
    """Fetch last_run timestamp from state table (safe for first run)."""

    if LOAD_TYPE == "FULL":
        return None

    #  FIRST RUN: table does not exist
    if not spark.catalog.tableExists(STATE_TABLE):
        print(f"[INFO] State table {STATE_TABLE} does not exist yet -> first run")
        return None

    rows = spark.sql(f"""
        SELECT last_run
        FROM {STATE_TABLE}
        WHERE pipeline = '{PIPELINE_NAME}'
        LIMIT 1
    """).collect()

    return rows[0]["last_run"] if rows else None


def ensure_state_table_exists():
    """Create pipeline_state table if it does not exist."""
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
            pipeline STRING,
            last_run STRING
        )
        USING DELTA
    """)


def update_last_run(ts: str) -> None:
    """Upsert last_run timestamp into state table."""
    spark.sql(f"""
        MERGE INTO {STATE_TABLE} t
        USING (SELECT '{PIPELINE_NAME}' AS pipeline, '{ts}' AS last_run) s
        ON t.pipeline = s.pipeline
        WHEN MATCHED THEN UPDATE SET t.last_run = s.last_run
        WHEN NOT MATCHED THEN INSERT *
    """)


# ============================================================
# QUERY
# ============================================================

def build_query(last_run: Optional[str]) -> Optional[str]:
    """Build TOPdesk query string."""
    if LOAD_TYPE == "FULL" or not last_run:
        return None
    return f"modificationDate>{last_run}"


# ============================================================
# HTTP FETCH
# ============================================================

def fetch_page(
    session: requests.Session,
    start: int,
    query: Optional[str],
) -> List[Dict]:
    """Fetch a single page with retry logic."""

    params = {
        "start": start,
        "page_size": PAGE_SIZE
    }

    if query:
        params["query"] = query

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(BASE_URL, params=params, timeout=REQUEST_TIMEOUT)

            if resp.status_code == 204:
                return []

            if resp.status_code in (200, 206):
                return resp.json()

            print(f"[WARN] HTTP {resp.status_code} at start={start} (attempt {attempt})")

        except ValueError:
            print(f"[WARN] Invalid JSON at start={start} (attempt {attempt})")

        except requests.RequestException as e:
            print(f"[WARN] Request error at start={start}: {e}")

        time.sleep(1.5 * attempt)

    print(f"[ERROR] Failed page start={start} after retries")
    return []


# ============================================================
# PARALLEL FETCH
# ============================================================

def fetch_all_parallel(query: Optional[str]) -> List[Dict]:
    """Fetch all incidents using parallel paging."""
    all_data: List[Dict] = []
    next_start = 0

    with requests.Session() as session:
        session.auth = AUTH

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

            while True:
                starts = [next_start + i * PAGE_SIZE for i in range(MAX_WORKERS)]

                futures = {
                    executor.submit(fetch_page, session, s, query): s
                    for s in starts
                }

                batch_empty = True

                for future in as_completed(futures):
                    data = future.result()

                    if data:
                        batch_empty = False
                        all_data.extend(data)

                        if len(data) < PAGE_SIZE:
                            return all_data

                if batch_empty:
                    break

                next_start += PAGE_SIZE * MAX_WORKERS
                print(f"[INFO] fetched={len(all_data)}")

    return all_data

# ===========================================================
# NEXT PAGINATION(SEQUENTIAL)
# ===========================================================

def fetch_all_sequential(query) -> List[Dict]:
    """Fetch all incidents using sequential paging."""
    url = BASE_URL
    if query:
        url = f"{BASE_URL}&query={query}"
    all_data = []
    
    with requests.Session() as session:
        session.auth = AUTH

        while url:
            resp = session.get(url, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 204:
                break
            if resp.status_code in (200, 206):
                print(f"[WARN] HTTP {resp.status_code} -> {url}")
                break
            
            payload = resp.json()

            results = payload.get("results", [])
            all_data.extend(results)
            print(f"[INFO] fetched={len(all_data)}")
            url = payload.get("next")
    
    return all_data
            

# ============================================================
# ROUTER (AUTO-DETECT PAGINATION)
# ============================================================

def fetch_all(query):
    """Detect pagination style and route to correct strategy"""

    with requests.Session() as session:
        session.auth = AUTH
        resp = session.get(BASE_URL, params={"start": 0, "page_size": 1}, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        sample = resp.json()

    if isinstance(sample, dict) and "next" in sample:
        print("[INFO] Using NEXT pagination (sequential)")
        return fetch_all_sequential(query)
    else:
        print("[INFO] Using OFFSET pagination (parallel)")
        return fetch_all_parallel(query)
    


# ============================================================
# WRITE TO BRONZE
# ============================================================

def write_to_bronze(incidents: List[Dict]) -> None:
    """Write raw incidents into bronze Delta table."""
    now = datetime.now(timezone.utc).isoformat()

    rows = [
        Row(
            raw_json=json.dumps(rec),
            ingestion_time=now
        )
        for rec in incidents
    ]

    df = spark.createDataFrame(rows)

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE)
    )

    print(f"[INFO] Written {len(incidents)} records to {BRONZE_TABLE}")


# ============================================================
# MAIN
# ============================================================

def main():

    print(f"[INFO] Running pipeline in {LOAD_TYPE} mode")

    #  Get state (safe for first run)
    last_run = get_last_run()

    #  Build query
    query = build_query(last_run)

    if query:
        print(f"[INFO] Filter: {query}")
    else:
        print("[INFO] FULL load (no filter)")

    # unified fetch
    records = fetch_all(query)

    print(f"[INFO] Total fetched: {len(records)}")



    if not records:
        print("[INFO] No new data - exiting")
        return

    # Write bronze
    write_to_bronze(records)

    # Ensure state table exists BEFORE merge
    ensure_state_table_exists()

    #  Update state
    new_last_run = datetime.now(timezone.utc).isoformat()
    update_last_run(new_last_run)

    print(f"[INFO] Updated last_run → {new_last_run}")


# ============================================================
# ENTRYPOINT
# ============================================================

if __name__ == "__main__":
    main()

In [0]:
import json
import time
import requests

from typing import Optional, List, Dict
from datetime import datetime, timezone
from pyspark.sql import Row
from concurrent.futures import ThreadPoolExecutor, as_completed

password = dbutils.secrets.get(scope="topdesk-scope", key="topdeskapi")
username = "topdeskapi"

# ================= CONFIG =================

CATALOG = "elixir"
BRONZE_SCHEMA = "brz"
META_SCHEMA = "meta"

#BASE_URL = "https://elixir.msf.org/tas/api/incidents"


AUTH = (username, password)

PAGE_SIZE = 100
MAX_WORKERS = 10
MAX_RETRIES = 3
REQUEST_TIMEOUT = (3, 10)

LOAD_TYPE = "FULL"   # "FULL" or "INCREMENTAL"

# PIPELINE_NAME = "topdesk_incidents"

# BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_incidents_raw"
STATE_TABLE  = f"{CATALOG}.{META_SCHEMA}."
BASE_URL = "https://elixir.msf.org/tas/api/incidents/timeregistrations?fields=timeSpent,operator.id,creationDate,parent.id,notes"
PIPELINE_NAME = "topdesk_incident_timeregistrations"
BRONZE_TABLE = "elixir.brz.topdesk_timeregistrations_raw"



# ============================================================
# STATE MANAGEMENT
# ============================================================

def get_last_run() -> Optional[str]:
    if LOAD_TYPE == "FULL":
        return None

    if not spark.catalog.tableExists(STATE_TABLE):
        print(f"[INFO] State table {STATE_TABLE} does not exist yet -> first run")
        return None

    rows = spark.sql(f"""
        SELECT last_run
        FROM {STATE_TABLE}
        WHERE pipeline = '{PIPELINE_NAME}'
        LIMIT 1
    """).collect()

    return rows[0]["last_run"] if rows else None


def ensure_state_table_exists():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
            pipeline STRING,
            last_run STRING
        )
        USING DELTA
    """)


def update_last_run(ts: str) -> None:
    spark.sql(f"""
        MERGE INTO {STATE_TABLE} t
        USING (SELECT '{PIPELINE_NAME}' AS pipeline, '{ts}' AS last_run) s
        ON t.pipeline = s.pipeline
        WHEN MATCHED THEN UPDATE SET t.last_run = s.last_run
        WHEN NOT MATCHED THEN INSERT *
    """)


# ============================================================
# QUERY
# ============================================================

def build_query(last_run: Optional[str]) -> Optional[str]:
    if LOAD_TYPE == "FULL" or not last_run:
        return None
    return f"modificationDate>{last_run}"


# ============================================================
# OFFSET PAGINATION (PARALLEL)
# ============================================================

def fetch_page(session: requests.Session, start: int, query: Optional[str]) -> List[Dict]:
    params = {
        "start": start,
        "page_size": PAGE_SIZE
    }

    if query:
        params["query"] = query

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(BASE_URL, params=params, timeout=REQUEST_TIMEOUT)

            if resp.status_code == 204:
                return []

            if resp.status_code in (200, 206):
                return resp.json()

            print(f"[WARN] HTTP {resp.status_code} at start={start} (attempt {attempt})")

        except ValueError:
            print(f"[WARN] Invalid JSON at start={start} (attempt {attempt})")

        except requests.RequestException as e:
            print(f"[WARN] Request error at start={start}: {e}")

        time.sleep(1.5 * attempt)

    print(f"[ERROR] Failed page start={start} after retries")
    return []


def fetch_all_parallel(query: Optional[str]) -> List[Dict]:
    all_data: List[Dict] = []
    next_start = 0

    with requests.Session() as session:
        session.auth = AUTH

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            while True:
                starts = [next_start + i * PAGE_SIZE for i in range(MAX_WORKERS)]

                futures = {
                    executor.submit(fetch_page, session, s, query): s
                    for s in starts
                }

                batch_empty = True

                for future in as_completed(futures):
                    data = future.result()

                    if data:
                        batch_empty = False
                        all_data.extend(data)

                        if len(data) < PAGE_SIZE:
                            return all_data

                if batch_empty:
                    break

                next_start += PAGE_SIZE * MAX_WORKERS
                print(f"[INFO] fetched={len(all_data)}")

    return all_data


# ============================================================
# NEXT PAGINATION (SEQUENTIAL)
# ============================================================

def fetch_all_sequential(query: Optional[str]) -> List[Dict]:
    url = BASE_URL
    if query:
        url = f"{BASE_URL}&query={query}"

    all_data: List[Dict] = []

    with requests.Session() as session:
        session.auth = AUTH

        while url:
            resp = session.get(url, timeout=REQUEST_TIMEOUT)

            if resp.status_code == 204:
                break

            if resp.status_code not in (200, 206):
                print(f"[WARN] HTTP {resp.status_code} → {url}")
                break

            payload = resp.json()

            results = payload.get("results", [])
            all_data.extend(results)

            print(f"[INFO] fetched={len(all_data)}")

            url = payload.get("next")

    return all_data


# ============================================================
# ROUTER (AUTO-DETECT PAGINATION)
# ============================================================

def fetch_all(query: Optional[str]) -> List[Dict]:
    """Detect pagination style and route to correct strategy."""

    with requests.Session() as session:
        session.auth = AUTH
        resp = session.get(BASE_URL, params={"start": 0, "page_size": 1}, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        sample = resp.json()

    if isinstance(sample, dict) and "next" in sample:
        print("[INFO] Using NEXT pagination (sequential)")
        return fetch_all_sequential(query)
    else:
        print("[INFO] Using OFFSET pagination (parallel)")
        return fetch_all_parallel(query)


# ============================================================
# WRITE TO BRONZE
# ============================================================

def write_to_bronze(records: List[Dict]) -> None:
    now = datetime.now(timezone.utc).isoformat()

    rows = [
        Row(
            raw_json=json.dumps(rec),
            ingestion_time=now
        )
        for rec in records
    ]

    df = spark.createDataFrame(rows)

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE)
    )

    print(f"[INFO] Written {len(records)} records to {BRONZE_TABLE}")


# ============================================================
# MAIN
# ============================================================

def main():
    print(f"[INFO] Running pipeline in {LOAD_TYPE} mode")

    last_run = get_last_run()
    query = build_query(last_run)

    if query:
        print(f"[INFO] Filter: {query}")
    else:
        print("[INFO] FULL load (no filter)")

    # unified fetch
    records = fetch_all(query)

    print(f"[INFO] Total fetched: {len(records)}")

    if not records:
        print("[INFO] No new data — exiting")
        return

    write_to_bronze(records)

    ensure_state_table_exists()

    new_last_run = datetime.now(timezone.utc).isoformat()
    update_last_run(new_last_run)

    print(f"[INFO] Updated last_run -> {new_last_run}")


# ============================================================
# ENTRYPOINT
# ============================================================

if __name__ == "__main__":
    main()

In [0]:
df = spark.table(BRONZE_TABLE)
display(df)

In [0]:
ENDPOINTS = [
    {
        "name": "incidents",
        "url": "https://elixir.msf.org/tas/api/incidents",
        "fields": None,
    },
    # {
    #     "name": "time_registrations",
    #     "url": "https://company/tas/api/incidents/timeregistrations",
    #     # this endpoint may or may not support fields
    #     "fields":"timeSpent,operator.id,creationDate,parent.id,notes",
    # },
    # {
    #     "name": "priorities",
    #     "url": "https://company/tas/api/changes",
    #     "fields": None,
    # },
]

In [0]:
import requests
import time
import json
from datetime import datetime, timezone
from typing import Optional, List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import Row

# ================= SECRETS =================

password = dbutils.secrets.get(scope="topdesk-scope", key="topdeskapi")
username = "topdeskapi"

# ================= CONFIG =================

CATALOG = "elixir"
BRONZE_SCHEMA = "brz"
META_SCHEMA = "meta"




AUTH = (username, password)

PAGE_SIZE = 100
MAX_WORKERS = 10
MAX_RETRIES = 3
REQUEST_TIMEOUT = 30

LOAD_TYPE = "INCREMENTAL"   # "FULL" or "INCREMENTAL"
STATE_TABLE  = f"{CATALOG}.{META_SCHEMA}.pipeline_state"

PIPELINE_NAME = "topdesk_ingestion"


In [0]:
import requests
import json
password = dbutils.secrets.get(scope="topdesk-scope", key="topdeskapi")
username = "topdeskapi"

In [0]:
ENDPOINTS = [
    {
        "name": "incidents",
        "url": "https://elixir.msf.org/tas/api/incidents",
        "fields": None,
        "incremental_field": "modificationDate",
        "bronze_table": "elixir.brz.topdesk_incidents_raw"
    },
    {
        "name": "time_registrations",
        "url": "https://elixir.msf.org/tas/api/incidents/timeregistrations",
        # this endpoint may or may not support fields
        "fields":"timeSpent,operator.id,creationDate,parent.id,notes,modificationDate",
        "incremental_field": "creationDate",
        "bronze_table": "elixir.brz.topdesk_timeregistrations_raw"
         
    },
    {
        "name": "priorities",
        "url": "https://elixir.msf.org/tas/api/incidents/priorities",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_incident_priorities_raw"
    },
]

In [0]:
def write_to_bronze(incidents: List[Dict], endpoint) -> None:
    """Write raw incidents into bronze Delta table."""
    bronze_table = endpoint["bronze_table"]
    now = datetime.now(timezone.utc).isoformat()

    rows = [
        Row(
            raw_json=json.dumps(rec),
            ingestion_time=now
        )
        for rec in incidents
    ]

    df = spark.createDataFrame(rows)

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_table)
    )

    print(f"[INFO] Written {len(incidents)} records to {bronze_table}")

In [0]:
def ensure_state_table_exists():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
            pipeline STRING,
            endpoint STRING,
            last_run STRING
        )
        USING DELTA
    """)


def get_last_run(endpoint_name: str):
    if LOAD_TYPE == "FULL":
        return None

    rows = spark.sql(f"""
        SELECT last_run
        FROM {STATE_TABLE}
        WHERE pipeline = '{PIPELINE_NAME}'
        AND endpoint = '{endpoint_name}'
        LIMIT 1
    """).collect()

    return rows[0]["last_run"] if rows else None


def update_last_run(endpoint_name: str, ts: str):
    spark.sql(f"""
        MERGE INTO {STATE_TABLE} t
        USING (
            SELECT '{PIPELINE_NAME}' AS pipeline,
                   '{endpoint_name}' AS endpoint,
                   '{ts}' AS last_run
        ) s
        ON t.pipeline = s.pipeline AND t.endpoint = s.endpoint
        WHEN MATCHED THEN UPDATE SET t.last_run = s.last_run
        WHEN NOT MATCHED THEN INSERT *
    """)


In [0]:
def build_query(last_run, endpoint):
    inc_field = endpoint.get("incremental_field")

    if LOAD_TYPE == "FULL" or not last_run or not inc_field:
        return None

    return f"{inc_field}>{last_run}"

In [0]:
def build_url(endpoint, query):
    base = endpoint["url"]

    params = []

    if endpoint.get("fields"):
        params.append(f"fields={endpoint['fields']}")

    if query:
        params.append(f"query={query}")

    if params:
        return base + "?" + "&".join(params)

    return base

In [0]:
def fetch_page(session, endpoint, start, query):
    params = {
        "start": start,
        "page_size": PAGE_SIZE
    }

    if query:
        params["query"] = query

    if endpoint.get("fields"):
        params["fields"] = endpoint["fields"]

    resp = session.get(endpoint["url"], params=params, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()

    return resp.json()

In [0]:
def stream_endpoint_to_bronze(endpoint, query):
    endpoint_name = endpoint["name"]
    inc_field = endpoint.get("incremental_field")

    total = 0
    batch = []
    max_ts = None

    # --- detect pagination type ---
    with requests.Session() as session:
        session.auth = AUTH
        probe = session.get(endpoint["url"], params={"start": 0, "page_size": 1}, timeout=REQUEST_TIMEOUT)
        probe.raise_for_status()
        sample = probe.json()

    is_next = isinstance(sample, dict) and "next" in sample

    print(f"[INFO] {endpoint_name} pagination: {'NEXT' if is_next else 'OFFSET'}")

    # =========================================================
    # NEXT PAGINATION
    # =========================================================
    if is_next:
        url = build_url(endpoint, query)

        with requests.Session() as session:
            session.auth = AUTH

            while url:
                resp = session.get(url, timeout=REQUEST_TIMEOUT)
                resp.raise_for_status()

                payload = resp.json()
                results = payload.get("results", [])

                for rec in results:
                    # track max timestamp safely
                    if LOAD_TYPE != "FULL" and inc_field:
                        mod = rec.get(inc_field)
                        if mod and (max_ts is None or mod > max_ts):
                            max_ts = mod

                    batch.append(rec)
                    total += 1

                if len(batch) >= 30000:
                    write_to_bronze(batch, endpoint)
                    batch.clear()

                print(f"[INFO] {endpoint_name} fetched={total}")

                url = payload.get("next")

    # =========================================================
    # OFFSET PAGINATION
    # =========================================================
    else:
        next_start = 0

        with requests.Session() as session:
            session.auth = AUTH

            while True:
                data = fetch_page(session, endpoint, next_start, query)

                if not data:
                    break

                for rec in data:
                    if LOAD_TYPE != "FULL" and inc_field:
                        mod = rec.get(inc_field)
                        if mod and (max_ts is None or mod > max_ts):
                            max_ts = mod

                    batch.append(rec)
                    total += 1

                if len(batch) >= 30000:
                    write_to_bronze(batch, endpoint)
                    batch.clear()

                if len(data) < PAGE_SIZE:
                    break

                next_start += PAGE_SIZE

                print(f"[INFO] {endpoint_name} fetched={total}")

    if batch:
        write_to_bronze(batch, endpoint)

    return total, max_ts

In [0]:
def main():
    """
    ============================================================
    TOPDESK INGESTION PIPELINE
    ============================================================

    This pipeline ingests multiple TOPdesk API endpoints into
    Bronze Delta tables.

    Features:
    - Supports FULL and INCREMENTAL loads
    - Handles multiple endpoints with different schemas
    - Supports both OFFSET and NEXT pagination styles
    - Writes data in batches for performance and stability
    - Tracks incremental state per endpoint

    Flow per endpoint:
        1. Read last_run from state table
        2. Build query filter (if incremental)
        3. Fetch data from API
        4. Write to Bronze in batches
        5. Update last_run checkpoint

    ============================================================
    """
    print(f"[INFO] Running pipeline in {LOAD_TYPE} mode")

    ensure_state_table_exists()

    for endpoint in ENDPOINTS:
        print("=" * 60)

        endpoint_name = endpoint["name"]
        print(f"[PIPELINE] {endpoint_name}")

        last_run = get_last_run(endpoint_name)

        query = build_query(last_run, endpoint)

        total, max_ts = stream_endpoint_to_bronze(endpoint, query)

        print(f"[DONE] {endpoint_name} -> {total} records")

        # update state only for incremental endpoints
        if endpoint.get("incremental_field") and max_ts:
            update_last_run(endpoint_name, max_ts)

In [0]:
if __name__ == "__main__":
    main()

In [0]:
import requests

BASE = "https://elixir.msf.org/tas/api/incidents/timeregistrations"
AUTH = (username, password)

tests = {
    "TEST_1_manual_encoded_with_ms": 
        "creationDate%3E2026-02-26T14:00:00.000%2B0000",

    "TEST_2_manual_encoded_no_ms": 
        "creationDate%3E2026-02-26T14:00:00%2B0000",
}

# -------- TEST 1 & 2 (manual URL) --------
for name, q in tests.items():
    url = f"{BASE}?start=0&page_size=100&query={q}"
    try:
        r = requests.get(url, auth=AUTH, timeout=10)
        print(f"{name}: status={r.status_code}")
        if r.status_code == 200:
            print(" -> records:", len(r.json().get("results", [])))
        else:
            print(" -> error:", r.text[:120])
    except Exception as e:
        print(f"{name}: EXCEPTION -> {e}")

# -------- TEST 3 (requests params encoding) --------
params = {
    "start": 0,
    "page_size": 100,
    "query": "creationDate>2026-02-26T14:00:00.000+0000"
}

try:
    r = requests.get(BASE, params=params, auth=AUTH, timeout=10)
    print("\nTEST_3_requests_params:", r.status_code)
    if r.status_code == 200:
        print(" -> records:", len(r.json().get("results", [])))
    else:
        print(" -> error:", r.text[:120])
except Exception as e:
    print("TEST_3_requests_params: EXCEPTION ->", e)

In [0]:
url1 = "https://elixir.msf.org/tas/api/incidents/timeregistrations?fields=timeSpent,operator.id,creationDate,notes"
url2 = "https://elixir.msf.org/tas/api/incidents/timeregistrations?fields=timeSpent,operator.id,creationDate,notes?query=creationDate>2026-02-25T00:00:00"

r1 = requests.get(url1, auth=(username, password)).json()
r2 = requests.get(url2, auth=(username, password)).json()

print(len(r1.get("results", [])), len(r2.get("results", [])))

In [0]:
r1

In [0]:
url = "https://elixir.msf.org/tas/api/incidents/timeregistrations?fields=creationDate,modificationDate&start=0&page_size=1"

resp = requests.get(url, auth=(username, password))
data = resp.json()

print(data["results"][0])

In [0]:
%sql
INSERT INTO elixir.meta.pipeline_state
(pipeline, endpoint, last_run)
VALUES (
  'topdesk_ingestion',
  'time_registrations',
  '2026-02-25T14:00:00.000+0000'
)

In [0]:
%sql
MERGE INTO elixir.meta.pipeline_state t

USING (
  SELECT
    'topdesk_ingestion' AS pipeline,
    'time_registrations' AS endpoint,
    '2026-02-26T14:00:00.000+0000' AS last_run
) s
ON t.pipeline = s.pipeline AND t.endpoint = s.endpoint
WHEN MATCHED THEN UPDATE SET t.last_run = s.last_run
WHEN NOT MATCHED THEN INSERT *